# Limpieza y normalización (3FN) del dataset de calidad del aire

Flujo completo:

```
raw (CSV en HDFS) → limpieza → normalización 3FN → processed (Parquet en HDFS)
                                                        ├─ locations, parametros, sensores → PostgreSQL
                                                        └─ mediciones                      → MongoDB
```

**Modelo objetivo**

| Tabla | Destino | Clave | Contenido |
|---|---|---|---|
| `locations` | PostgreSQL | `location_id` | estaciones de medición |
| `parametros` | PostgreSQL | `parameter_id` | contaminante y su unidad |
| `sensores` | PostgreSQL | `sensor_id` | qué parámetro mide cada sensor y en qué ubicación |
| `mediciones` | MongoDB | `(sensor_id, datetime_utc)` | una lectura por sensor y hora, con fecha UTC y local |

Requisito: contenedores arriba (`docker compose up -d`).

## 1. Conexión a Spark (Spark Connect)

In [1]:
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.remote("sc://localhost:15002").getOrCreate()
spark.range(5).show()

+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+



## 2. Carga del dato crudo
Cambiá `DATASET` por el nombre de tu carpeta en HDFS.

In [2]:
DATASET = "contaminacion"
RAW = f"hdfs://namenode:9000/data/raw/{DATASET}/"
OUT = f"hdfs://namenode:9000/data/processed/{DATASET}"

# datetime_local se lee como texto 
ESQUEMA = ("location_id INT, location_name STRING, sensor_id INT, parameter STRING, value DOUBLE, "
           "unit STRING, datetime_utc TIMESTAMP, datetime_local STRING")
sdf = spark.read.csv(RAW, header=True, schema=ESQUEMA)
sdf.printSchema()
sdf.show(5)

root
 |-- location_id: integer (nullable = true)
 |-- location_name: string (nullable = true)
 |-- sensor_id: integer (nullable = true)
 |-- parameter: string (nullable = true)
 |-- value: double (nullable = true)
 |-- unit: string (nullable = true)
 |-- datetime_utc: timestamp (nullable = true)
 |-- datetime_local: string (nullable = true)

+-----------+-------------+---------+---------+-----+-----+-------------------+--------------------+
|location_id|location_name|sensor_id|parameter|value| unit|       datetime_utc|      datetime_local|
+-----------+-------------+---------+---------+-----+-----+-------------------+--------------------+
|       2139|        Essex| 12387120|     pm10| 17.0|µg/m³|2025-03-05 18:00:00|2025-03-05T13:00:...|
|       2139|        Essex| 12387120|     pm10| 13.0|µg/m³|2025-03-05 19:00:00|2025-03-05T14:00:...|
|       2139|        Essex| 12387120|     pm10| 13.0|µg/m³|2025-03-05 20:00:00|2025-03-05T15:00:...|
|       2139|        Essex| 12387120|     pm10|  8

## 3. Exploración inicial
Tamaño, nulos por columna y duplicados.

In [3]:
print("Filas:", sdf.count(), "| Columnas:", len(sdf.columns))

nulos = sdf.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in sdf.columns])
nulos.toPandas().T.rename(columns={0: "nulos"})

Filas: 587449 | Columnas: 8


,nulos
location_id,0
location_name,0
sensor_id,0
parameter,0
value,4063
unit,0
datetime_utc,0
datetime_local,0


In [4]:
print("Duplicados exactos:", sdf.count() - sdf.dropDuplicates().count())
sdf.describe().show()

Duplicados exactos: 0
+-------+-----------------+------------------+-----------------+---------+----------------+------+--------------------+
|summary|      location_id|     location_name|        sensor_id|parameter|           value|  unit|      datetime_local|
+-------+-----------------+------------------+-----------------+---------+----------------+------+--------------------+
|  count|           587449|            587449|           587449|   587449|          583386|587449|              587449|
|   mean|867.8892908150325|              NULL|565458.1834440096|     NULL|5.48836670403468|  NULL|                NULL|
| stddev|582.2387615865738|              NULL|2352915.600930294|     NULL|8.75622888762202|  NULL|                NULL|
|    min|              448|  Boston - Roxbury|              779|       o3|            -4.0|   ppm|2016-03-06T14:00:...|
|    max|             2139|McMillan Reservoir|         12394931|     pm25|           456.0| µg/m³|2026-11-04T17:00:...|
+-------+---------

## 4. Limpieza

Primero se limpia la tabla plana original (una fila = una lectura). Cada paso tiene su justificación y muestra cuántas filas afecta, para que el proceso sea auditable.

### 4.1 Normalizar nombres de columnas
Minúsculas, sin espacios ni símbolos. Evita errores de escritura y problemas con PostgreSQL.

In [5]:
import re

def normalizar(nombre):
    return re.sub(r"[^a-z0-9]+", "_", nombre.strip().lower()).strip("_")

df = sdf.toDF(*[normalizar(c) for c in sdf.columns])
n_inicial = df.count()
print("Filas iniciales:", n_inicial)
df.columns

Filas iniciales: 587449


['location_id',
 'location_name',
 'sensor_id',
 'parameter',
 'value',
 'unit',
 'datetime_utc',
 'datetime_local']

### 4.2 Limpiar texto
Se eliminan espacios sobrantes y se unifica a minúsculas el nombre del parámetro (`pm10`, `pm25`, `o3`). Si hubiera `PM10` y `pm10`, la normalización los trataría como parámetros distintos.

In [6]:
df = (df
      .withColumn("location_name", F.trim("location_name"))
      .withColumn("parameter", F.lower(F.trim("parameter")))
      .withColumn("unit", F.trim("unit")))

df.select("parameter", "unit").distinct().show()

+---------+-----+
|parameter| unit|
+---------+-----+
|     pm10|µg/m³|
|     pm25|µg/m³|
|       o3|  ppm|
+---------+-----+



### 4.3 Tipos de datos
Se fijan los tipos explícitamente en vez de depender de `inferSchema`: ids enteros, `value` decimal y fechas como `timestamp`.

`datetime_local` viene como `2025-03-05T13:00:00-05:00`. Se le quita el desfase y se guarda la **hora local tal cual** (13:00). Así no coincide con la UTC y sirve para buscar por fecha y hora local.

In [19]:
df = (df
      .withColumn("location_id", F.col("location_id").cast("int"))
      .withColumn("sensor_id", F.col("sensor_id").cast("int"))
      .withColumn("value", F.col("value").cast("double"))
      .withColumn("datetime_utc", F.col("datetime_utc").cast("timestamp")))
df.printSchema()

root
 |-- location_id: integer (nullable = true)
 |-- location_name: string (nullable = true)
 |-- sensor_id: integer (nullable = true)
 |-- parameter: string (nullable = true)
 |-- value: double (nullable = true)
 |-- unit: string (nullable = true)
 |-- datetime_utc: timestamp (nullable = true)
 |-- datetime_local: timestamp (nullable = true)



### 4.4 Nulos
- Una lectura **sin valor** no aporta información medible, así que se elimina.
- Una fila sin `sensor_id` o `datetime_utc` no se puede identificar (son su clave), así que también se elimina.

En la exploración se vio que `value` tiene unos 4 000 nulos y el resto de columnas ninguno.

In [17]:
claves = ["location_id", "location_name", "sensor_id", "parameter", "unit", "datetime_utc"]
antes = df.count()
df = df.dropna(subset=claves + ["value"])
print(f"Filas eliminadas por nulos: {antes - df.count():,}")

Filas eliminadas por nulos: 0


### 4.5 Valores imposibles
Una concentración de contaminante no puede ser negativa. Los valores negativos (por ejemplo -4) son ruido o descalibración del sensor, no mediciones reales, y se eliminan.

Los valores **altos** no se eliminan automáticamente. Un pico de 456 puede ser un evento real de contaminación, no un error. Se inspeccionan por parámetro antes de decidir.

In [18]:
antes = df.count()
df = df.filter(F.col("value") >= 0)
print(f"Filas eliminadas por valores negativos: {antes - df.count():,}")

(df.groupBy("parameter", "unit")
   .agg(F.min("value").alias("min"), F.expr("percentile_approx(value, 0.5)").alias("mediana"),
        F.expr("percentile_approx(value, 0.99)").alias("p99"), F.max("value").alias("max"))
   .show())

Filas eliminadas por valores negativos: 0
+---------+-----+---+-------+-----+-----+
|parameter| unit|min|mediana|  p99|  max|
+---------+-----+---+-------+-----+-----+
|     pm25|µg/m³|0.0|    6.0| 26.4|358.3|
|     pm10|µg/m³|0.0|   13.0| 51.0|456.0|
|       o3|  ppm|0.0|  0.028|0.066|0.131|
+---------+-----+---+-------+-----+-----+



### 4.6 Duplicados
Una lectura queda identificada por **sensor + instante**. Si aparece dos veces la misma combinación se conserva una sola. No se buscan duplicados por fila completa, porque dos filas con la misma clave y distinto valor también serían inconsistentes.

In [10]:
antes = df.count()
df = df.dropDuplicates(["sensor_id", "datetime_utc"])
print(f"Filas eliminadas por duplicados: {antes - df.count():,}")
print(f"Filas tras la limpieza: {df.count():,} (de {n_inicial:,})")

Filas eliminadas por duplicados: 0
Filas tras la limpieza: 583,324 (de 587,449)


### 4.7 Verificar las dependencias funcionales
Antes de normalizar hay que **comprobar** que las dependencias en las que se basa el diseño se cumplen en los datos. Cada una debe dar 0 violaciones:

| Dependencia | Significa |
|---|---|
| `location_id → location_name` | cada ubicación tiene un único nombre |
| `sensor_id → location_id` | un sensor está en una sola ubicación |
| `sensor_id → parameter` | un sensor mide un solo contaminante |
| `parameter → unit` | cada contaminante tiene una sola unidad |

In [11]:
dependencias = [
    ("location_id", "location_name"),
    ("sensor_id", "location_id"),
    ("sensor_id", "parameter"),
    ("parameter", "unit"),
]

for det, dep in dependencias:
    violaciones = (df.groupBy(det).agg(F.countDistinct(dep).alias("n")).filter("n > 1").count())
    print(f"{det} -> {dep}: {violaciones} violaciones")

location_id -> location_name: 0 violaciones
sensor_id -> location_id: 0 violaciones
sensor_id -> parameter: 0 violaciones
parameter -> unit: 0 violaciones


## 5. Normalización hasta 3FN

La tabla plana repite en cada una de las ~580 000 filas el nombre de la ubicación, el parámetro y la unidad. Eso causa redundancia y anomalías de actualización.

| Forma normal | Regla | Cómo se cumple aquí |
|---|---|---|
| **1FN** | valores atómicos, sin grupos repetidos | cada celda tiene un único valor |
| **2FN** | sin dependencias parciales de la clave | la clave de la lectura es `(sensor_id, datetime_utc)`; `value` depende de ambas. Lo que depende solo de `sensor_id` (ubicación, parámetro) se mueve a `sensores` |
| **3FN** | sin dependencias transitivas | `sensor_id → parameter → unit` es transitiva, así que `unit` va en `parametros`. Y `sensor_id → location_id → location_name` también, así que el nombre va en `locations` |

### 5.1 `locations`
Una fila por ubicación.

In [12]:
locations = df.select("location_id", "location_name").distinct().orderBy("location_id")
locations.show()

+-----------+------------------+
|location_id|     location_name|
+-----------+------------------+
|        448|  Boston - Roxbury|
|        626|      Bronx - IS52|
|        739|McMillan Reservoir|
|       2139|             Essex|
+-----------+------------------+



### 5.2 `parametros`
Una fila por contaminante, con su unidad. Se genera `parameter_id` como clave numérica. Se ordena por nombre para que los ids sean siempre los mismos al repetir el proceso.

In [13]:
from pyspark.sql.window import Window

parametros = (df.select("parameter", "unit").distinct()
              .withColumn("parameter_id", F.row_number().over(Window.orderBy("parameter")).cast("int"))
              .select("parameter_id", "parameter", "unit"))
parametros.show()

c:\Users\alear\AppData\Local\Programs\Python\Python313\Lib\site-packages\pyspark\sql\connect\expressions.py:1091: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+------------+---------+-----+
|parameter_id|parameter| unit|
+------------+---------+-----+
|           1|       o3|  ppm|
|           2|     pm10|µg/m³|
|           3|     pm25|µg/m³|
+------------+---------+-----+



### 5.3 `sensores`
Una fila por sensor, con la ubicación donde está y el parámetro que mide. El parámetro se referencia por `parameter_id`, no por texto.

In [14]:
sensores = (df.select("sensor_id", "location_id", "parameter").distinct()
            .join(parametros.select("parameter_id", "parameter"), on="parameter")
            .select("sensor_id", "location_id", "parameter_id")
            .orderBy("sensor_id"))
sensores.show()

+---------+-----------+------------+
|sensor_id|location_id|parameter_id|
+---------+-----------+------------+
|      779|        448|           1|
|      783|        448|           3|
|     1098|        626|           1|
|     1099|        626|           2|
|     1102|        626|           3|
|     3119|        739|           1|
|     3121|        739|           2|
|     3123|        739|           3|
|     3846|       2139|           1|
|  2381110|        448|           2|
| 12387120|       2139|           2|
| 12394931|       2139|           3|
+---------+-----------+------------+



### 5.4 `mediciones`
Solo queda lo que depende de la clave completa: sensor, instante (UTC) y valor. Ubicación, parámetro y unidad se obtienen a través de `sensor_id`.


In [15]:
mediciones = df.select("sensor_id", "datetime_utc", "value")
print(f"mediciones: {mediciones.count():,} filas")
mediciones.show(5)

mediciones: 583,324 filas
+---------+-------------------+-----+
|sensor_id|       datetime_utc|value|
+---------+-------------------+-----+
| 12387120|2025-03-08 01:00:00| 13.0|
| 12387120|2025-03-08 07:00:00| 10.0|
| 12387120|2025-03-08 08:00:00| 12.0|
| 12387120|2025-03-08 09:00:00| 11.0|
| 12387120|2025-03-08 15:00:00|  6.0|
+---------+-------------------+-----+
only showing top 5 rows


### 5.5 Validar la integridad
Antes de guardar se comprueba que las claves sean únicas y que no haya referencias huérfanas. PostgreSQL lo exigirá con sus claves foráneas, y MongoDB no lo valida, así que conviene verificarlo aquí.

In [16]:
def es_unica(tabla, cols):
    return tabla.count() == tabla.select(*cols).distinct().count()

print("PK locations.location_id      :", es_unica(locations, ["location_id"]))
print("PK parametros.parameter_id    :", es_unica(parametros, ["parameter_id"]))
print("PK sensores.sensor_id         :", es_unica(sensores, ["sensor_id"]))
print("PK mediciones (sensor, fecha) :", es_unica(mediciones, ["sensor_id", "datetime_utc"]))

print("FK sensores -> locations      :", sensores.join(locations, "location_id", "left_anti").count(), "huérfanos")
print("FK sensores -> parametros     :", sensores.join(parametros, "parameter_id", "left_anti").count(), "huérfanos")
print("FK mediciones -> sensores     :", mediciones.join(sensores, "sensor_id", "left_anti").count(), "huérfanos")

PK locations.location_id      : True
PK parametros.parameter_id    : True
PK sensores.sensor_id         : True
PK mediciones (sensor, fecha) : True
FK sensores -> locations      : 0 huérfanos
FK sensores -> parametros     : 0 huérfanos
FK mediciones -> sensores     : 0 huérfanos


## 6. Guardar el resultado limpio (Parquet en HDFS)
El Parquet es la capa limpia y la fuente de verdad. Si hay que recargar una base, se hace desde acá sin repetir la limpieza.

In [20]:
tablas = {"locations": locations, "parametros": parametros, "sensores": sensores, "mediciones": mediciones}

for nombre, tabla in tablas.items():
    tabla.write.mode("overwrite").parquet(f"{OUT}/{nombre}")
    print(nombre, "->", spark.read.parquet(f"{OUT}/{nombre}").count(), "filas")

locations -> 4 filas
parametros -> 3 filas
sensores -> 12 filas
mediciones -> 583324 filas


## 7. Cargar las tablas estructuradas en PostgreSQL

Las tablas ya existen con sus claves primarias y foráneas (las crea `db/init.sql` al levantar el contenedor), así que Spark solo inserta filas. El orden importa: primero las tablas referenciadas (`locations`, `parametros`) y después `sensores`.

Para **repetir la carga** hay que vaciar antes las tablas, todas juntas por las claves foráneas:
```
docker exec postgres psql -U postgres -d smartcities -c "TRUNCATE sensores, parametros, locations"
```

In [21]:
PG_URL = "jdbc:postgresql://postgres:5432/smartcities"
PG_PROPS = {"user": "postgres", "password": "postgres", "driver": "org.postgresql.Driver"}

for nombre in ["locations", "parametros", "sensores"]:
    (spark.read.parquet(f"{OUT}/{nombre}").write.jdbc(PG_URL, nombre, mode="append", properties=PG_PROPS))
    n = spark.read.jdbc(PG_URL, nombre, properties=PG_PROPS).count()
    print(f"{nombre}: {n} filas en PostgreSQL")

locations: 4 filas en PostgreSQL
parametros: 3 filas en PostgreSQL
sensores: 12 filas en PostgreSQL


## 8. Cargar las mediciones en MongoDB

`mediciones` es la tabla grande y de series temporales (~580 000 documentos), con campos simples: `sensor_id`, `datetime_utc`, `datetime_local`, `value`. Cada documento es una lectura.

In [22]:
MONGO_URI = "mongodb://mongo:27017"

(spark.read.parquet(f"{OUT}/mediciones")
      .write.format("mongodb")
      .option("connection.uri", MONGO_URI)
      .option("database", "smartcities")
      .option("collection", "mediciones")
      .mode("overwrite").save())

n = (spark.read.format("mongodb")
     .option("connection.uri", MONGO_URI)
     .option("database", "smartcities")
     .option("collection", "mediciones").load().count())
print(f"mediciones: {n:,} documentos en MongoDB")

mediciones: 583,324 documentos en MongoDB
